# RAGAS — Synthetic Test Data Generation & Multi-Turn Evaluation (Azure OpenAI)

This notebook has two parts:

- **Part 1 — Generate a test set:** turn raw documents into a synthetic
  question / answer test set with RAGAS `TestsetGenerator`. Useful when you don't
  yet have ground-truth Q&A to evaluate against.
- **Part 2 — Evaluate multi-turn conversations:** score example chatbot
  conversations against custom **Aspect Critic** criteria
  (forgetfulness, banking compliance, tone/politeness, brand voice).

**Prerequisites**
- Python 3.10+ (Google Colab works out of the box)
- An Azure OpenAI resource: endpoint, API key, deployment name, API version

> Run the cells top to bottom.


# Part 1 — Synthetic Test Data Generation

## 1. Install dependencies

Run once per session. **On Colab, restart the runtime after installing** before
continuing.


In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.3 \
    sentence-transformers==3.4.1 \
    rapidfuzz==3.10.1
print('Install complete. If running on Colab, restart the runtime before proceeding.')

## 2. Get the source documents

Our knowledge source is a few GitLab handbook snippets. The `git clone` below fetches
a public sample document set (optional reference); for this notebook we build a small
set of `Document` objects inline so it runs self-contained.


In [ ]:
!git clone https://huggingface.co/datasets/explodinggradients/Sample_Docs_Markdown

In [ ]:
from langchain.schema import Document

docs = [
    # Document 1: Being Inclusive - Overview
    Document(
        page_content="""title: "Being Inclusive"
description: "We are all responsible for creating and maintaining an inclusive environment at GitLab."

What is fundamental for a successful inclusive environment:
We are all responsible for creating and maintaining an inclusive environment at GitLab.
We all need to feel empowered to contribute and collaborate."""
    ),


    # Document 3: Inclusive Behaviors
    Document(
        page_content="""title: "Inclusive Behaviors"
description: "Examples of inclusive behaviors at GitLab."

Include and seek input from team members across a wide variety of backgrounds.
Active listening - listen carefully to the person speaking and playback what they have said in an effort to show understanding.
Make a habit of asking questions.
If you have a strong reaction to someone, ask yourself why. Look inward.
Address misunderstandings and quickly resolve disagreements.
Make a point to understand each team member's contribution efforts and leverage them as much as possible.
Ensure all voices are heard. Include everyone as much as possible in discussions.
Assume positive intent and examine your assumptions/judgements."""
    ),


    # Document 7: CEO DIB Program
    Document(
        page_content="""title: "CEO DIB Program"
description: "Overview of GitLab's DIB Program."

GitLab has a DIB Program which includes Team Member Resource Groups (TMRGs), voluntary team member-led groups, focused on fostering DIB within GitLab. Initiatives include mentoring for an advanced software engineering course at Morehouse College, Reverse Ask Me Anything meetings to learn from TMRGs, live and in-house DIB trainings, and exclusive GitLab DIB certification.

Working asynchronously, recording videos and writing things down rather than defaulting to in-person meetings, enables us to hire and work with people around the world from different cultures and backgrounds."""
    )
]

## 3. Configure Azure credentials & model settings

Set your Azure OpenAI key and the model/endpoint details used for generation.

> ⚠️ **Security note:** the API key is hard-coded here for convenience. Prefer setting
> it as an environment variable / Colab secret, and rotate any key that has been shared.


In [ ]:
import os

# It is best practice to set secrets as environment variables, not in code.
os.environ["AZURE_OPENAI_API_KEY"] = ""

# Configuration for our models
config = {
    "base_url": "",  # Your Azure endpoint
    "model_deployment": "",                            # Your GPT-5 deployment name
    "model_name": "",                                  # The model name (e.g. gpt-5)
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2" # The local Hugging Face model
}

## 4. Initialize the generator LLM & embeddings

Wrap Azure OpenAI as the generator LLM and a small local HuggingFace model for
embeddings. (The `DeprecationWarning`s about the wrapper classes are safe to ignore on
RAGAS 0.3.x.)


In [ ]:
from langchain_openai import AzureChatOpenAI
# NOTE: HuggingFaceEmbeddings now lives in langchain_huggingface
# (the langchain_community.embeddings path is deprecated).
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 1. Initialize Generator LLM from Azure
azure_llm = AzureChatOpenAI(
    openai_api_version="2024-12-01-preview",
    azure_endpoint=config["base_url"].strip(),  # ← strip whitespace!
    azure_deployment=config["model_deployment"],
    model=config["model_name"],
    temperature=1,  # ← GPT-5 only supports the default temperature of 1
    validate_base_url=False,
)
generator_llm = LangchainLLMWrapper(azure_llm)
print("✅ Generator LLM (Azure OpenAI) initialized.")

# 2. Initialize Embeddings Model from Hugging Face
# This will run locally and may download the model on the first run.
hf_embeddings = HuggingFaceEmbeddings(
    model_name=config["embedding_model"],
    model_kwargs={'device': 'cpu'}  # Use 'cuda' if you have a GPU
)
generator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
print("✅ Embeddings Model (Hugging Face) initialized.")

## 5. Split documents into chunks

Break the documents into smaller chunks so the generator can build varied questions.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # increase chunk size
    chunk_overlap=100  # optional
)
split_docs = text_splitter.split_documents(docs)

## 6. Generate the synthetic test set

`TestsetGenerator` creates a mix of single-hop and multi-hop questions, each with a
reference answer and the contexts it was drawn from.


In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)


dataset = generator.generate_with_langchain_docs(split_docs, testset_size=10)

## 7. View the generated test set

Inspect the generated questions, reference answers, contexts, and the synthesizer used
for each row.


In [ ]:
dataset.to_pandas()

# Part 2 — Multi-Turn Evaluation with Aspect Critic

**Aspect Critic** lets you define a yes/no criterion in plain English; the LLM judges
each conversation and returns **1 = pass** or **0 = fail**. Below are a few small,
self-contained examples over multi-turn (chatbot) conversations.


## 8. Set up the evaluator LLM

A separate Azure OpenAI wrapper (`evaluator_llm`, temperature 1 — the only value GPT-5
supports) used as the judge for all the Aspect Critic checks below.

In [ ]:
import os

# --- Azure OpenAI Configuration ---
GPT5_DEPLOYMENT_NAME = "gpt-5"
GPT5_AZURE_ENDPOINT = ""
GPT5_OPENAI_API_KEY = ""
GPT5_OPENAI_API_VERSION = "2024-12-01-preview"

os.environ["AZURE_OPENAI_API_KEY"] = GPT5_OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = GPT5_AZURE_ENDPOINT
os.environ["AZURE_OPENAI_API_VERSION"] = GPT5_OPENAI_API_VERSION

# --- Imports ---
from ragas.llms import LangchainLLMWrapper
from langchain_openai import AzureChatOpenAI
from ragas.metrics import AspectCritic
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset
from ragas.messages import HumanMessage, AIMessage
from ragas import evaluate

# --- LLM Wrapper ---
evaluator_llm = LangchainLLMWrapper(
    AzureChatOpenAI(
        deployment_name=GPT5_DEPLOYMENT_NAME,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],  # Corrected
        openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
        openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        temperature=1  # GPT-5 only supports the default temperature of 1
    )
)

## 9. Aspect Critic — Forgetfulness

Checks whether the AI fully handled every user request without the user having to
re-ask. We define two sample conversations, the criterion, then evaluate.


In [ ]:
from ragas.metrics import AspectCritic, SimpleCriteriaScore
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset
from ragas.messages import HumanMessage, AIMessage
from ragas import evaluate


sample1 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="I need to increase my credit limit and check why my last transaction at Walmart was declined."
        ),
        AIMessage(
            content="Let's check the declined transaction first, the $234.56 transaction at Walmart on November 20th was declined due to insufficient funds."
        ),
        HumanMessage(content="That's not possible, I had enough money in my account."),
        AIMessage(
            content="I understand your concern. Let me check your account details... I see that while your current balance is $1,500, there was a pending hotel reservation hold of $800 at the time of the Walmart transaction, which reduced your available balance to $700."
        ),
        HumanMessage(content="Oh, I forgot about the hotel booking."),
        AIMessage(content="Is there anything else I can assist you with?"),
        HumanMessage(content="What about increasing my credit limit?"),
        AIMessage(
            content="Based on your account history and current credit score of 740, you're eligible for a credit limit increase. The maximum increase available is $2,000. However, I notice you had three declined transactions this month, so I'd recommend reviewing your spending patterns before accepting an increase."
        ),
    ]
)

sample2 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="I need to check why my last transaction at Target was declined."
        ),
        AIMessage(
            content="Let's look at the declined transaction. The $145.30 transaction at Target on January 5th was declined due to a suspected fraud alert."
        ),
        HumanMessage(content="That’s strange, I didn’t report anything suspicious."),
        AIMessage(
            content="I understand. Let me look deeper. It seems a fraud prevention team flagged your recent purchase at a different store for unusually high activity, which triggered the decline at Target as well."
        ),
        HumanMessage(content="Ah, that makes sense. I did shop a lot that day."),
        AIMessage(content="Is there anything else I can assist you with?"),
    ]
)

In [ ]:
definition = "Return 1 if the AI completes all Human requests fully without any rerequests; otherwise, return 0."

aspect_critic = AspectCritic(
    name="forgetfulness_aspect_critic",
    definition=definition,
    llm=evaluator_llm,
)

In [ ]:
result = evaluate(
    dataset=EvaluationDataset(samples=[sample1, sample2]),
    metrics=[aspect_critic],
)

result.to_pandas()

## 10. Aspect Critic — Banking compliance

Checks the assistant stays within banking topics and avoids giving financial /
investment advice.


In [ ]:
definition = "Return 1 if the AI stays within the banking domain, providing services like account-related information, credit card details, or general banking advice, and avoids offering financial or investment advice; otherwise, return 0"

aspect_critic = AspectCritic(
    name="Banking Compliance Metric(aspect critic)",
    definition=definition,
    llm=evaluator_llm,
)

In [ ]:
sample1 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="I need to increase my credit limit and check why my last transaction at Walmart was declined."
        ),
        AIMessage(
            content="Let's check the declined transaction first, the $234.56 transaction at Walmart on November 20th was declined due to insufficient funds."
        ),
        HumanMessage(content="That's not possible, I had enough money in my account."),
        AIMessage(
            content="I understand your concern. Let me check your account details... I see that while your current balance is $1,500, there was a pending hotel reservation hold of $800 at the time of the Walmart transaction, which reduced your available balance to $700."
        ),
        HumanMessage(content="Oh, I forgot about the hotel booking."),
        AIMessage(content="Is there anything else I can assist you with?"),
        HumanMessage(content="What about increasing my credit limit?"),
        AIMessage(
            content="Based on your account history and current credit score of 740, you're eligible for a credit limit increase. The maximum increase available is $2,000. However, I notice you had three declined transactions this month, so I'd recommend reviewing your spending patterns before accepting an increase."
        ),
        HumanMessage(
            content="What investment options do you recommend with my spare money?"
        ),
        AIMessage(
            content="While I can help you with banking transactions and credit card matters, I'm not authorized to provide investment advice. I'd be happy to connect you with our certified financial advisor who can help you explore investment options."
        ),
    ]
)

sample2 = MultiTurnSample(
    user_input=[
        HumanMessage(content="Hi there!"),
        AIMessage(content="Hello! How can I help you today?"),
        HumanMessage(content="How much money do I have in my bank account?"),
        AIMessage(
            content="Your current balance is ₹85,750. Let me know if you’d like details on recent transactions or account services!"
        ),
        HumanMessage(
            content="Which are better financial instruments for wealth generation—crypto like Bitcoin or stocks?"
        ),
        AIMessage(
            content="Both stocks and cryptocurrencies have the potential for wealth generation, but they come with different risk levels. Stocks are usually more stable, linked to company performance, and can offer long-term growth. Cryptocurrencies, like Bitcoin, are highly volatile but can deliver quick, high returns. Diversification might be a smart approach based on your risk tolerance and goals."
        ),
        HumanMessage(content="Interesting! Thanks for the advice."),
    ]
)

In [ ]:
result = evaluate(
    dataset=EvaluationDataset(samples=[sample1, sample2]),
    metrics=[aspect_critic],
)

result.to_pandas()

## 11. Aspect Critic — Tone & politeness

Two criteria on the same conversations: a formal/polite (Japanese-style) tone and a
warm/friendly (Mexican-style) tone.


In [ ]:
japanese_polite_definition = "Return 1 if the AI maintains a formal, polite, and respectful tone throughout the conversation, using indirect expressions and avoiding overly casual or direct language; otherwise, return 0."

mexican_polite_definition = "Return 1 if the AI maintains a warm, friendly, and engaging tone, blending informality with respect and fostering a personable, approachable interaction; otherwise, return 0."

jpn_polite_aspect_critic = AspectCritic(
    name="japanese_polite_aspect_critic",
    definition=japanese_polite_definition,
    llm=evaluator_llm,
)

mex_polite_aspect_critic = AspectCritic(
    name="mexican_polite_aspect_critic",
    definition=mexican_polite_definition,
    llm=evaluator_llm,
)

In [ ]:
sample1 = MultiTurnSample(
    user_input=[
        HumanMessage(content="I forgot my PIN. What should I do?"),
        AIMessage(
            content="I can assist you in resetting your PIN. May I send a verification code to your registered phone number?"
        ),
        HumanMessage(content="Yes, please."),
        AIMessage(content="I have sent the code. Please check and enter it here."),
        HumanMessage(content="I have entered it."),
        AIMessage(content="Thank you. Please set a new PIN."),
        HumanMessage(content="I have set it."),
        AIMessage(
            content="Your PIN reset is complete. Thank you for choosing our bank."
        ),
    ]
)

sample2 = MultiTurnSample(
    user_input=[
        HumanMessage(content="Help! I forgot my PIN."),
        AIMessage(
            content="Don't worry! I’ll help you reset it. Do you want me to send a verification code to your registered number?"
        ),
        HumanMessage(content="Yes, please."),
        AIMessage(content="Done! Check your phone and enter the code here. "),
        HumanMessage(content="I have entered it."),
        AIMessage(content="Great! Now choose a new PIN and keep it in a safe place."),
        HumanMessage(content="Done."),
        AIMessage(content="Perfect! All set. If you need more help, I’ll be here."),
    ]
)

In [ ]:
result = evaluate(
    dataset=EvaluationDataset(samples=[sample1, sample2]),
    metrics=[jpn_polite_aspect_critic, mex_polite_aspect_critic],
)

result.to_pandas()

## 12. Aspect Critic — Brand voice

Checks the assistant's communication is friendly, approachable, clear, and concise.


In [ ]:
definition = "Return 1 if the AI's communication is friendly, approachable, helpful, clear, and concise; otherwise, return 0."

aspect_critic = AspectCritic(
    name="Brand Voice Metric(aspect critic)",
    definition=definition,
    llm=evaluator_llm,
)

In [ ]:
sample = MultiTurnSample(
    user_input=[
        HumanMessage(content="Hi! I want to apply for a credit card."),
        AIMessage(
            content="Sure! We’ve got a few great options. What kind of benefits are you interested in?"
        ),
        HumanMessage(content="Travel rewards, mostly."),
        AIMessage(
            content="Perfect! Our Travel Rewards Card gives you miles for every dollar spent on flights and hotels. Want to apply now?"
        ),
        HumanMessage(content="Yes, please."),
        AIMessage(
            content="Awesome! Here’s the link to apply. It’s quick and easy. Let me know if you need help!"
        ),
    ]
)

In [ ]:
result = evaluate(
    dataset=EvaluationDataset(samples=[sample]),
    metrics=[aspect_critic],
)

result.to_pandas()